# [REFILLCARE] RefillCare: Medication Refill Reminder System
## End-to-End Walkthrough: Phases 1, 2, and 3

**Author:** RefillCare Core Pipeline  
**Date:** 2026-09-07  

---

### [TARGET] Project Overview
RefillCare is an intelligent pharmacy refill reminder system built to predict when patients will need chronic medication refills based on historical purchasing intervals and active pharmaceutical ingredient (SALT) patterns.

### [DOCS] Notebook Contents
1. **Phase 1 — Exploratory Data Analysis & Discovery:** Raw transaction inspection, customer identity resolution, shared-phone analysis, and SALT catalog mapping.
2. **Phase 2 — Data Pipeline & History Creation:** Data cleaning, customer ID sanitization, invoice line aggregation, SALT enrichment, and chronological interval calculation.
3. **Phase 3 — Leakage-Safe Feature Engineering & Baseline:** Supervised target formulation ($y = t_{i+1} - t_i$), 36 engineered features, temporal splitting, and historical median baseline evaluation.

## [TOOLS] Step 0: Imports & Universal Path Resolver

In [28]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

current_dir = Path(__file__).resolve().parent if '__file__' in locals() else Path.cwd()
project_root = current_dir.resolve()
while project_root.parent != project_root and not (project_root / 'refillcare' / '__init__.py').exists():
    project_root = project_root.parent
if (project_root / 'refillcare' / '__init__.py').exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

def find_file(relative_path: str) -> Path:
    candidates = [
        project_root / relative_path,
        Path.cwd() / relative_path,
        Path('..') / relative_path,
        Path('../..') / relative_path,
        Path(r'C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder') / relative_path,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0]
print(f'[PASS] Project Root resolved: {project_root}')


[PASS] Project Root resolved: C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder


---
# 🔍 Phase 1: Exploratory Data Analysis & Source Inspection

Phase 1 analyzed the raw pharmacy point-of-sale dataset (`customer_data_fields.csv`) and master catalog (`SALT WISE ITEMS.xlsx`).

### Key Discoveries from Phase 1:
- **895,557 raw transaction rows** across ~5.8 years (`2020-12-24` to `2026-08-31`).
- **Identity Resolution:** `customerId` (28,413 unique) is the unique patient identifier. Phone numbers (`MOBILE_NO`) are shared across family members (136 shared phones involving 341 customers) and must NOT be used as customer identity.
- **Date Parsing:** Dates are formatted as `DD/MM/YYYY`, requiring `dayfirst=True`.
- **Duplicate Lines:** Multiple rows per `(customerId + invoice_number + itemId)` represent split batches on the same visit and must be aggregated.

In [29]:
# 1.1 Load and inspect raw transactions
raw_csv_path = find_file('data/refillcare/customer_data_fields.csv')
print(f'Loading from: {raw_csv_path}')

if raw_csv_path.exists():
    raw_sample = pd.read_csv(raw_csv_path, nrows=5000)
    print(f'[PASS] Loaded {len(raw_sample):,} rows sample')
    display(raw_sample[['invoice_date', 'invoice_number', 'customerId', 'customerName', 'itemId', 'itemName', 'quantity', 'MOBILE_NO']].head(5))
else:
    print(f'[FAIL] File not found at: {raw_csv_path}')

Loading from: C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder\data\refillcare\customer_data_fields.csv
[PASS] Loaded 5,000 rows sample


,invoice_date,invoice_number,customerId,customerName,itemId,itemName,quantity,MOBILE_NO
0,01/06/2025,SB/391,PHARMA HUBB A12 NGOS_9966473474,PHARMA HUBB A12 NGOS,14532,EUKROMA KJ CREAM,1,9966473474
1,01/06/2025,SB/391,PHARMA HUBB A12 NGOS_9966473474,PHARMA HUBB A12 NGOS,7107,DERMADEW LITE SOAP,2,9966473474
2,01/06/2025,SB/391,PHARMA HUBB A12 NGOS_9966473474,PHARMA HUBB A12 NGOS,9298,IMXIA XL SERUM 6 60ML,1,9966473474
3,01/06/2025,SB/392,PHARMA HUBB A18 BN REDDY,PHARMA HUBB A18 BN REDDY,7670,SLEM LYS TAB,1,
4,01/06/2025,SB/392,PHARMA HUBB A18 BN REDDY,PHARMA HUBB A18 BN REDDY,7670,SLEM LYS TAB,2,


In [30]:
# 1.2 Inspect SALT Master Catalog
salt_excel_path = find_file('data/refillcare/SALT WISE ITEMS.xlsx')
print(f'Loading from: {salt_excel_path}')

if salt_excel_path.exists():
    salt_sample = pd.read_excel(salt_excel_path, sheet_name='Rate List', skiprows=5, nrows=10, engine='openpyxl')
    print(f'[PASS] SALT Master Catalog Sample (skiprows=5):')
    display(salt_sample[['Code', 'Item Name', 'PACK', 'SALT', 'CATEGORY', 'ITEMCAT']].head(5))
else:
    print(f'[FAIL] File not found at: {salt_excel_path}')

Loading from: C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder\data\refillcare\SALT WISE ITEMS.xlsx
[PASS] SALT Master Catalog Sample (skiprows=5):


,Code,Item Name,PACK,SALT,CATEGORY,ITEMCAT
0,14199,CLOP E CREAM,20 GM,CLOBETASOL-20GM,OINTMENTS/CREAMS/GEL/PASTE,OINTMENTS/CREAMS/GEL/PASTE
1,17880,VENUSIA MOISTURIZING CREAM,100GM,,"ARTICLES FOR FUNFAIR, TABLE OR PARLOUR GAMES, ...","ARTICLES FOR FUNFAIR, TABLE OR PARLOUR GAMES, ..."
2,35861,AKDERM AF BODY WASH,100ML,,BODY WASH,BODY WASH
3,33883,BEPANTHEN SENSI CONTROL BODY WASH,100ML,,BODY WASH,BODY WASH
4,34471,DOVE RELAXING B/W,1000ML,,BODY WASH,BODY WASH


---
# [CLEAN] Phase 2: Data Pipeline, History Creation & SALT Enrichment

Phase 2 built a reusable, tested pipeline (`refillcare.data`) that executes:
1. **Cleaning & Sanitization:** Removes exact duplicates, drops null `mfgDate`, and sanitizes noisy characters from `customerId`.
2. **Invoice Aggregation:** Sums quantities and amounts for duplicate invoice lines, creating 1 purchase event per visit.
3. **SALT Enrichment:** Left joins master catalog active ingredients on `Code` $\leftrightarrow$ `itemCode`.
4. **History & Interval Math:** Constructs chronological timelines per `customerId + itemId` and computes backward-looking intervals.

In [31]:
# 2.1 Load Phase 2 Processed Purchase History
history_parquet_path = find_file('data/refillcare/processed/purchase_history.parquet')
print(f'Loading from: {history_parquet_path}')

if history_parquet_path.exists():
    history_df = pd.read_parquet(history_parquet_path)
    print(f'[PASS] Total Purchase Events: {len(history_df):,}')
    print(f'[PASS] Unique Customer-Medicine Histories: {history_df.groupby(["customerId", "itemId"]).ngroups:,}')
    display(history_df[['customerId', 'customerName', 'itemId', 'itemName', 'invoice_date', 'quantity', 'purchase_seq', 'days_since_previous_purchase', 'salt_composition']].head(8))
else:
    print(f'[FAIL] File not found at: {history_parquet_path}')

Loading from: C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder\data\refillcare\processed\purchase_history.parquet
[PASS] Total Purchase Events: 817,804
[PASS] Unique Customer-Medicine Histories: 194,324


,customerId,customerName,itemId,itemName,invoice_date,quantity,purchase_seq,days_since_previous_purchase,salt_composition
0,9884880372,9884880372,1196,ACROFY LOTIN,2025-07-22,1,1,NaN,NaN
1,9884880372,9884880372,18446,SYNTRAN SB 130 CAP,2025-07-22,1,1,NaN,ITRACONAZOLE 130MG
2,9884880372,9884880372,26740,OLESOFT LITE MOIST GEL,2025-07-22,1,1,NaN,NaN
3,9884880372,9884880372,4153,DERIVA C MS GEL,2025-07-22,1,1,NaN,ADAPALENE-0.10W/W+CLINDAMYCIN-1W/W
4,A,A,11002,EVEREADY CELLS AAA,2026-06-07,3,1,NaN,NaN
5,A,A,6955,SASTRY BALM,2025-12-06,1,1,NaN,NaN
6,A,A,7815,ZANDU BALM 25ML,2025-10-06,1,1,NaN,NaN
7,A ADITHYA VISHAL_9989715815,A ADITHYA VISHAL,11623,VICKS COUGH DROPS,2025-09-17,1,1,NaN,NaN


In [32]:
# 2.2 Inspect Customer Refill Interval Statistics
intervals = history_df['days_since_previous_purchase'].dropna()

print('--- Refill Interval Summary ---')
print(f'Total Intervals:    {len(intervals):,}')
print(f'Median Interval:    {intervals.median():.1f} days')
print(f'Mean Interval:      {intervals.mean():.2f} days')
print(f'Std Deviation:      {intervals.std():.2f} days')
recurring_pct = ((intervals >= 15) & (intervals <= 120)).sum() / len(intervals) * 100
print(f'Recurring Cycles (15-120d): {recurring_pct:.2f}%')

--- Refill Interval Summary ---
Total Intervals:    157,750
Median Interval:    29.0 days
Mean Interval:      42.35 days
Std Deviation:      47.30 days
Recurring Cycles (15-120d): 69.60%


---
# [LAUNCH] Phase 3: Feature Engineering & Supervised Learning Dataset

Phase 3 constructed the leakage-safe training dataset (`refillcare.features`) and established baseline benchmark performance.

### Supervised Learning Formulation:
- **Target:** `target_days_until_next_purchase` = $t_{i+1} - t_i$.
- **Final Purchase Rule:** Event $N$ in every history has no known next purchase and is strictly excluded from training targets.
- **36 Leakage-Safe Features:** Historical expanding median/mean/std, quantity ratios, calendar features, and medicine/SALT attributes.

In [33]:
# 3.1 Load Processed Supervised Datasets
train_path = find_file('data/refillcare/processed/train.parquet')
val_path = find_file('data/refillcare/processed/validation.parquet')
test_path = find_file('data/refillcare/processed/test.parquet')

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f'[PASS] Train Set:      {len(train_df):,} rows ({train_df.invoice_date.min().date()} to {train_df.invoice_date.max().date()})')
print(f'[PASS] Validation Set: {len(val_df):,} rows ({val_df.invoice_date.min().date()} to {val_df.invoice_date.max().date()})')
print(f'[PASS] Test Set:       {len(test_df):,} rows ({test_df.invoice_date.min().date()} to {test_df.invoice_date.max().date()})')

# Display engineered features sample
display(train_df[['customerId', 'itemId', 'invoice_date', 'purchase_count_so_far', 'historical_interval_median', 'avg_historical_quantity', 'purchase_month', 'target_days_until_next_purchase']].head(8))

[PASS] Train Set:      469,583 rows (2020-12-24 to 2026-04-30)
[PASS] Validation Set: 12,821 rows (2026-05-01 to 2026-06-30)
[PASS] Test Set:       7,556 rows (2026-07-01 to 2026-08-31)


,customerId,itemId,invoice_date,purchase_count_so_far,historical_interval_median,avg_historical_quantity,purchase_month,target_days_until_next_purchase
0,A ADITHYA VISHAL_9989715815,492,2025-06-06,1,NaN,2.000000,6,22.0
1,A ADITHYA VISHAL_9989715815,492,2025-06-28,2,22.0,2.000000,6,18.0
2,A ADITHYA VISHAL_9989715815,492,2025-07-16,3,20.0,2.333333,7,13.0
3,A ADITHYA VISHAL_9989715815,492,2025-07-29,4,18.0,2.000000,7,13.0
4,A ADITHYA VISHAL_9989715815,492,2025-08-11,5,15.5,2.000000,8,14.0
5,A ADITHYA VISHAL_9989715815,492,2025-08-25,6,14.0,1.833333,8,9.0
6,A ADITHYA VISHAL_9989715815,492,2025-09-03,7,13.5,1.857143,9,14.0
7,A ADITHYA VISHAL_9989715815,492,2025-09-22,9,13.5,1.555556,9,7.0


In [34]:
# 3.2 Inspect Phase 3 Quality Report JSON
report_json_path = find_file('data/refillcare/processed/phase3_quality_report.json')
if report_json_path.exists():
    with open(report_json_path, 'r') as f:
        report = json.load(f)
    
    print('=== Phase 3 Baseline Benchmark Results ===')
    print(f'Fallback Training Median: {report["baseline_evaluation"]["fallback_median_days"]} days')
    print('\nValidation Metrics:')
    for k, v in report["baseline_evaluation"]["validation"].items():
        if k != 'target_summary':
            print(f'  - {k}: {v}')
            
    print('\nTest Metrics:')
    for k, v in report["baseline_evaluation"]["test"].items():
        if k != 'target_summary':
            print(f'  - {k}: {v}')

=== Phase 3 Baseline Benchmark Results ===
Fallback Training Median: 30.0 days

Validation Metrics:
  - count: 22621
  - mae: 19.26
  - rmse: 33.86
  - r2: -1.5211
  - within_1_day_pct: 10.9
  - within_3_days_pct: 23.14
  - within_7_days_pct: 40.18

Test Metrics:
  - count: 11618
  - mae: 17.28
  - rmse: 35.5
  - r2: -7.8387
  - within_1_day_pct: 12.6
  - within_3_days_pct: 26.28
  - within_7_days_pct: 44.89


---
## 🏁 Summary & Phase 4 Roadmap

| Phase | Deliverable | Status |
|---|---|---|
| **Phase 1** | Data Analysis & Quality Profiling (`docs/PHASE_1_REFILLCARE_DATA_ANALYSIS.md`) | [PASS] Complete |
| **Phase 2** | Data Cleaning, Aggregation & History Pipeline (`refillcare.data`) | [PASS] Complete |
| **Phase 3** | Leakage-Safe Feature Engineering & Baseline (`refillcare.features`) | [PASS] Complete |
| **Phase 4** | **Machine Learning Model Training (XGBoost / LightGBM)** | 🔜 Ready to Begin |

### Next Step: Phase 4
Train Gradient Boosted Decision Trees (XGBoost / LightGBM) and Quantile Regressors on `train.parquet`, validate on `validation.parquet`, and evaluate on `test.parquet` to beat the baseline MAE of 19.2 days.